In [1]:
import pandas as pd
import numpy as np
import torch
import os
import re
import nltk

download_dir = os.path.join('..', 'data', 'nltk_data')
os.makedirs(download_dir, exist_ok=True)
nltk.download('punkt_tab', download_dir=download_dir)
nltk.data.path.append(download_dir)

[nltk_data] Downloading package punkt_tab to ../data/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [46]:
os.chdir('/content/drive/My Drive/NLP-Letters-V2/notebooks/')
df1 = pd.read_csv('../data/letters_2021.csv')
df2 = pd.read_csv('../data/sentence_sets_trimmed.csv', encoding='mac-roman')
df2 = df2[df2['full_text_tokens'] > 10]

# Preprocess Data

In [47]:
degender_mapping = {
    r'(?:^|\b|[^\w\s]+)mr(?:\b|[^\w\s]+|$)': ' mx ',
    r'(?:^|\b|[^\w\s]+)mrs(?:\b|[^\w\s]+|$)': ' mx ',
    r'(?:^|\b|[^\w\s]+)ms(?:\b|[^\w\s]+|$)': ' mx ',
    r'(?:^|\b|[^\w\s]+)miss(?:\b|[^\w\s]+|$)': ' mx ',
    r'(?:^|\b|[^\w\s]+)mister(?:\b|[^\w\s]+|$)': ' mx ',
    r'(?:^|\b|[^\w\s]+)him(?:\b|[^\w\s]+|$)': ' them ',
    r'(?:^|\b|[^\w\s]+)her(?:\b|[^\w\s]+|$)': ' them ',
    r'(?:^|\b|[^\w\s]+)hers(?:\b|[^\w\s]+|$)': ' theirs ',
    r'(?:^|\b|[^\w\s]+)he(?:\b|[^\w\s]+|$)': ' they ',
    r'(?:^|\b|[^\w\s]+)he\'s(?:\b|[^\w\s]+|$)': ' they\'re ',
    r'(?:^|\b|[^\w\s]+)she\'s(?:\b|[^\w\s]+|$)': ' they\'re ',
    r'(?:^|\b|[^\w\s]+)he\’s(?:\b|[^\w\s]+|$)': ' they\'re ',
    r'(?:^|\b|[^\w\s]+)she\’s(?:\b|[^\w\s]+|$)': ' they\'re ',
    r'(?:^|\b|[^\w\s]+)his(?:\b|[^\w\s]+|$)': ' their ',
    r'(?:^|\b|[^\w\s]+)her(?:\b|[^\w\s]+|$)': ' their ',
    r'(?:^|\b|[^\w\s]+)she(?:\b|[^\w\s]+|$)': ' they ',
    r'(?:^|\b|[^\w\s]+)himself(?:\b|[^\w\s]+|$)': ' themself ',
    r'(?:^|\b|[^\w\s]+)herself(?:\b|[^\w\s]+|$)': ' themself ',
    r'(?:^|\b|[^\w\s]+)man(?:\b|[^\w\s]+|$)': ' person ',
    r'(?:^|\b|[^\w\s]+)men(?:\b|[^\w\s]+|$)': ' persons ',
    r'(?:^|\b|[^\w\s]+)woman(?:\b|[^\w\s]+|$)': ' person ',
    r'(?:^|\b|[^\w\s]+)women(?:\b|[^\w\s]+|$)': ' persons ',
    r'(?:^|\b|[^\w\s]+)man\'s(?:\b|[^\w\s]+|$)': ' person\'s ',
    r'(?:^|\b|[^\w\s]+)men\'s(?:\b|[^\w\s]+|$)': ' person\'s ',
    r'(?:^|\b|[^\w\s]+)woman\'s(?:\b|[^\w\s]+|$)': ' person\'s ',
    r'(?:^|\b|[^\w\s]+)women\'s(?:\b|[^\w\s]+|$)': ' person\'s ',
    r'(?:^|\b|[^\w\s]+)gentleman(?:\b|[^\w\s]+|$)': ' person ',
    r'(?:^|\b|[^\w\s]+)lady(?:\b|[^\w\s]+|$)': ' person ',
    r'(?:^|\b|[^\w\s]+)gentleman\'s(?:\b|[^\w\s]+|$)': ' person\'s ',
    r'(?:^|\b|[^\w\s]+)lady\'s(?:\b|[^\w\s]+|$)': ' person\'s ',
    r'(?:^|\b|[^\w\s]+)first_name(?:\b|[^\w\s]+|$)': ' identifier ',
    r'(?:^|\b|[^\w\s]+)last_name(?:\b|[^\w\s]+|$)': ' identifier ',
    r'(?:^|\b|[^\w\s]+)middle_name(?:\b|[^\w\s]+|$)': ' identifier ',
    r'(?:^|\b|[^\w\s]+)identifier(?:\b|[^\w\s]+|$)': ' identifier ',
    r'(?:^|\b|[^\w\s]+)possible_identifier(?:\b|[^\w\s]+|$)': ' identifier '
}

### First Dataset

In [48]:
df1['sentences'] = df1['LETTERTEXT'].apply(nltk.sent_tokenize)

In [49]:
identifier_pattern = r'(?:^|\b|[^\w\s]+)(first_name|identifier|last_name|middle_name)(?:\b|[^\w\s]+|$)'
pronoun_pattern = r"(?:^|\b|[^\w\s]+)(he|she|him|himself|herself|her|his|hers|he’s|she’s|he's|she's)(?:\b|[^\w\s]+|$)"

In [50]:
def classify_sentence(sentence):
    if re.search(identifier_pattern, sentence, re.IGNORECASE):
        return 's1'
    elif re.search(pronoun_pattern, sentence, re.IGNORECASE):
        return 's2'
    else:
        return 's3'

In [51]:
df1['s1'] = None
df1['s2'] = None
df1['s3'] = None
df1['s1_s2'] = None

for idx, row in df1.iterrows():
    s1_sentences = []
    s2_sentences = []
    s3_sentences = []
    s1_s2_sentences = []

    for sentence in row['sentences']:
        category = classify_sentence(sentence)
        if category == 's1':
            s1_sentences.append(sentence)
            s1_s2_sentences.append(sentence)
        elif category == 's2':
            s2_sentences.append(sentence)
            s1_s2_sentences.append(sentence)
        else:
            s3_sentences.append(sentence)

    df1.at[idx, 's1'] = ' '.join(s1_sentences)
    df1.at[idx, 's2'] = ' '.join(s2_sentences)
    df1.at[idx, 's3'] = ' '.join(s3_sentences)
    df1.at[idx, 's1_s2'] = ' '.join(s1_s2_sentences)

In [52]:
df1['full_text'] = df1['LETTERTEXT'].astype(str)
df1['full_text'] = df1['full_text'].str.lower()
df1['s1'] = df1['s1'].str.lower()
df1['s2'] = df1['s2'].str.lower()
df1['s3'] = df1['s3'].str.lower()
df1['s1_s2'] = df1['s1_s2'].str.lower()

In [53]:
df1['full_text'] = df1['full_text'].replace(degender_mapping, regex=True)
df1['s1'] = df1['s1'].replace(degender_mapping, regex=True)
df1['s2'] = df1['s2'].replace(degender_mapping, regex=True)
df1['s3'] = df1['s3'].replace(degender_mapping, regex=True)
df1['s1_s2'] = df1['s1_s2'].replace(degender_mapping, regex=True)

In [54]:
df1['s1_s2'][0]

"it is my pleasure to write a letter of recommendation for  identifier , who has applied for a residency position with your program.  their  pleasing personality and sincere dedication in patient care make  their  a wonderful candidate. i first met  identifier  during  their  hospice and palliative medicine rotation at jamaica hospital medical center, jamaica, new york, while doing  their  family medicine core. as part of the team,  identifier  took part in evaluating and assessing patients under our consult service. prior to seeing patients,  they  diligently read patient charts to prepare  themself  for the day and proved to be a valuable asset to our clinical team.  they  took great pride in improving  their  clinical knowledge and skills.  they  demonstrated initiative and dedication in providing care to patients and their families on a daily basis, and especially during end of life. one of my most memorable interactions with  identifier  was during our consultation for a patient w

In [55]:
# df1.to_csv('../data/letters_2021_processed.csv', index=False)

In [38]:
# df1.to_csv('../data/letters_2021_processed_with_gender.csv', index=False)

### Second Dataset

In [56]:
df2['sentences'] = df2['TEXT'].apply(nltk.sent_tokenize)

In [57]:
identifier_pattern = r'(?:^|\b|[^\w\s]+)(first_name|identifier|last_name|middle_name)(?:\b|[^\w\s]+|$)'
pronoun_pattern = r"(?:^|\b|[^\w\s]+)(he|she|him|himself|herself|her|his|hers|he’s|she’s|he's|she's)(?:\b|[^\w\s]+|$)"

In [58]:
def classify_sentence(sentence):
    if re.search(identifier_pattern, sentence, re.IGNORECASE):
        return 's1'
    elif re.search(pronoun_pattern, sentence, re.IGNORECASE):
        return 's2'
    else:
        return 's3'

In [59]:
df2['s1'] = None
df2['s2'] = None
df2['s3'] = None
df2['s1_s2'] = None

for idx, row in df2.iterrows():
    s1_sentences = []
    s2_sentences = []
    s3_sentences = []
    s1_s2_sentences = []

    for sentence in row['sentences']:
        category = classify_sentence(sentence)
        if category == 's1':
            s1_sentences.append(sentence)
            s1_s2_sentences.append(sentence)
        elif category == 's2':
            s2_sentences.append(sentence)
            s1_s2_sentences.append(sentence)
        else:
            s3_sentences.append(sentence)

    df2.at[idx, 's1'] = ' '.join(s1_sentences)
    df2.at[idx, 's2'] = ' '.join(s2_sentences)
    df2.at[idx, 's3'] = ' '.join(s3_sentences)
    df2.at[idx, 's1_s2'] = ' '.join(s1_s2_sentences)

In [60]:
df2['full_text'] = df2['TEXT'].astype(str)
df2['full_text'] = df2['full_text'].str.lower()
df2['s1'] = df2['s1'].str.lower()
df2['s2'] = df2['s2'].str.lower()
df2['s3'] = df2['s3'].str.lower()
df2['s1_s2'] = df2['s1_s2'].str.lower()

In [61]:
df2['full_text'] = df2['full_text'].replace(degender_mapping, regex=True)
df2['s1'] = df2['s1'].replace(degender_mapping, regex=True)
df2['s2'] = df2['s2'].replace(degender_mapping, regex=True)
df2['s3'] = df2['s3'].replace(degender_mapping, regex=True)
df2['s1_s2'] = df2['s1_s2'].replace(degender_mapping, regex=True)

In [62]:
df2['s1_s2'][0]

" identifier   identifier  .  identifier  waived  their  right to review this letter .  identifier  received a clinical score of outstanding , a shelf board score of satisfactory , and an overall core im clerkship score of good .  identifier  received these comments on  their  core im clerkship : ¬¢ great documentation with accurate and well thought out assessments . ‚äî attending e  identifier  performed well on the rotation ; specific strengths included actively seeking opportunities for patient care , using strong communication strategies to engender trust with patients , asking for continuous feedback and seeking opportunities to educate team ( opportunities infections in hiv ) . - attending e it was a pleasure working with  identifier  on wards .  identifier 's a team player , gets along well with other staff and patients and family .  identifier  helped with team work and was active member of the team .  their  presentations were detailed oriented , write ups were well organized 

In [63]:
# df2.to_csv('../data/sentence_sets_trimmed_processed.csv', index=False)

In [45]:
# df2.to_csv('../data/sentence_sets_trimmed_processed_with_gender.csv', index=False)

# Combine Data

In [ ]:
df1 = df1[['s1_s2', 'full_text', 'LETTER_GENDER']]
df1 = df1.rename(columns={'LETTER_GENDER':'label'})

In [ ]:
df2 = df2[['s1_s2', 'full_text', 'applicant_gender']]
df2 = df2.rename(columns={'applicant_gender':'label'})

In [ ]:
df = pd.concat([df1, df2], ignore_index=True)

In [ ]:
gender_label_mapping = {
    'F':0,
    'female':0,
    'M':1,
    'male':1
}

In [ ]:
df['label'] = df['label'].replace(gender_label_mapping)

<ipython-input-61-cc45885305bc>:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['label'] = df['label'].replace(gender_label_mapping)


In [ ]:
df['s1_s2'][0]

"it is my pleasure to write a letter of recommendation for  identifier , who has applied for a residency position with your program.  their  pleasing personality and sincere dedication in patient care make  their  a wonderful candidate. i first met  identifier  during  their  hospice and palliative medicine rotation at jamaica hospital medical center, jamaica, new york, while doing  their  family medicine core. as part of the team,  identifier  took part in evaluating and assessing patients under our consult service. prior to seeing patients,  they  diligently read patient charts to prepare  themself  for the day and proved to be a valuable asset to our clinical team.  they  took great pride in improving  their  clinical knowledge and skills.  they  demonstrated initiative and dedication in providing care to patients and their families on a daily basis, and especially during end of life. one of my most memorable interactions with  identifier  was during our consultation for a patient w

In [ ]:
# df.to_csv('../data/df_sentences_without_gender.csv', index=False)